# Channel Attention with Six Parameters: SE and ECA in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/channel_attention_se_eca.ipynb)

A convolution produces a stack of channels and treats every one of them as equally important. Channel attention adds a cheap second pass that looks at what each channel found across the whole image and rescales it. Squeeze-and-Excitation does that with a two-layer bottleneck; ECA argues the bottleneck is the problem and replaces it with a single 1-D convolution.

This notebook builds both in NumPy first, checks them against the PyTorch versions to seven decimal places, then runs the ablation that decides whether either is worth having. The interesting part is section 4: the obvious way to compare three architectures does not compare three architectures.

Everything runs on a CPU in a few minutes. The training budget is smaller than the companion post's, so the numbers are noisier, which section 5 puts to use.

Companion post: [Channel Attention with Six Parameters: SE and ECA in PyTorch](https://sesen.ai/blog/channel-attention-squeeze-excitation-eca)

## 1. FashionMNIST

Ten classes of clothing at 28x28, harder than MNIST and still small enough that a convnet trains on a CPU in under a minute. The whole dataset goes into memory as one tensor, which is several times faster than a DataLoader at this size.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

torch.set_num_threads(4)

N_TRAIN, EPOCHS, BATCH, LR = 8000, 6, 128, 3e-3
CLASSES = ("T-shirt", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot")


def load_fashion_mnist(n_train=N_TRAIN):
    tr = datasets.FashionMNIST("./data", train=True, download=True)
    te = datasets.FashionMNIST("./data", train=False, download=True)

    def pack(ds):
        x = ds.data.float().div_(255.0).unsqueeze(1)
        return (x - 0.2860) / 0.3530, ds.targets.clone()

    xtr, ytr = pack(tr)
    xte, yte = pack(te)
    return (xtr[:n_train], ytr[:n_train]), (xte, yte)


(Xtr, ytr), (Xte, yte) = load_fashion_mnist()
print(f"train {len(ytr)}, test {len(yte)}, image {tuple(Xtr.shape[1:])}")

## 2. Both blocks, in NumPy

"Channel attention" sounds like it needs machinery. It does not. Each block is a pooling step, a small function, and a multiply:

- **squeeze**: average each channel's whole feature map down to one number
- **excite**: turn those C numbers into C gates in (0, 1)
- **scale**: multiply each channel by its own gate

Only the middle step differs. SE routes the C numbers through a bottleneck of width C/r and back, which costs `2 * C * C/r` weights and grows with the square of C. ECA slides one 1-D convolution of width k along the channel axis, which costs k weights, and k is 3, 5 or 7 whatever C is.

The last cell checks the NumPy versions against the PyTorch modules. They agree to float32 precision, so the four lines above are the computation rather than a picture of it.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def se_numpy(x, W1, b1, W2, b2):
    """Squeeze-and-Excitation, in four lines of NumPy.

    squeeze : average each channel's whole feature map down to one number
    excite  : two dense layers with a bottleneck of width C/r, then a sigmoid
    scale   : multiply every spatial position of a channel by its own gate
    """
    s = x.mean(axis=(2, 3))                        # (B, C)
    h = np.maximum(s @ W1.T + b1, 0.0)             # (B, C/r)
    g = sigmoid(h @ W2.T + b2)                     # (B, C)
    return x * g[:, :, None, None], g


def eca_numpy(x, w):
    """Efficient Channel Attention. Same squeeze, gate from one 1-D convolution.

    The same k weights are used at every channel, so a channel's gate depends
    only on itself and its k-1 nearest neighbours in the channel ordering.
    Nothing here grows with C.
    """
    s = x.mean(axis=(2, 3))                        # (B, C)
    k = len(w)
    sp = np.pad(s, ((0, 0), (k // 2, k // 2)))
    windows = np.lib.stride_tricks.sliding_window_view(sp, k, axis=1)
    g = sigmoid(windows @ w)                       # (B, C)
    return x * g[:, :, None, None], g


class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        g = torch.sigmoid(self.fc2(F.relu(self.fc1(x.mean(dim=(2, 3))))))
        return x * g[:, :, None, None]


def adaptive_k(channels, gamma=2, b=1):
    """ECA's kernel-size rule: the nearest odd number to (log2(C) + b) / gamma."""
    t = int(abs((math.log2(channels) + b) / gamma))
    return t if t % 2 else t + 1


class EfficientChannelAttention(nn.Module):
    def __init__(self, channels, k=None):
        super().__init__()
        self.k = k or adaptive_k(channels)
        self.conv = nn.Conv1d(1, 1, self.k, padding=self.k // 2, bias=False)

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        g = torch.sigmoid(self.conv(s[:, None, :])).squeeze(1)
        return x * g[:, :, None, None]


# the NumPy versions are the same computation, not an illustration of it
xs = np.random.default_rng(0).normal(size=(4, 64, 7, 7)).astype(np.float32)
se, eca = SqueezeExcitation(64).eval(), EfficientChannelAttention(64).eval()
with torch.no_grad():
    want_se, want_eca = se(torch.from_numpy(xs)).numpy(), eca(torch.from_numpy(xs)).numpy()
got_se, _ = se_numpy(xs, *[p.detach().numpy() for p in
                           (se.fc1.weight, se.fc1.bias, se.fc2.weight, se.fc2.bias)])
got_eca, _ = eca_numpy(xs, eca.conv.weight.detach().numpy().reshape(-1))
print(f"SE  NumPy vs torch: {np.abs(got_se - want_se).max():.2e}")
print(f"ECA NumPy vs torch: {np.abs(got_eca - want_eca).max():.2e}")
print(f"\nadaptive kernel size by channel count: "
      f"{ {c: adaptive_k(c) for c in (16, 64, 256, 1024)} }")

## 3. One backbone, three variants

Two convolutional blocks, global average pool, linear head. The attention block sits at the end of each block, after the convolutions and before the downsample, which is where SE-ResNet puts it.

Look at what the third column costs.

In [ ]:
def make_attention(variant, channels, **kw):
    if variant == "none":
        return nn.Identity()
    if variant == "SE":
        return SqueezeExcitation(channels, reduction=kw.get("reduction", 16))
    if variant == "ECA":
        return EfficientChannelAttention(channels, k=kw.get("k"))
    raise ValueError(variant)


class Block(nn.Module):
    def __init__(self, cin, cout, variant, **kw):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.attn = make_attention(variant, cout, **kw)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        return F.max_pool2d(self.attn(x), 2)


class SmallCNN(nn.Module):
    def __init__(self, variant="none", widths=(32, 64), **kw):
        super().__init__()
        c1, c2 = widths
        self.b1 = Block(1, c1, variant, **kw)
        self.b2 = Block(c1, c2, variant, **kw)
        self.head = nn.Linear(c2, 10)

    def forward(self, x):
        return self.head(self.b2(self.b1(x)).mean(dim=(2, 3)))


def attention_params(m):
    return sum(p.numel() for n, p in m.named_parameters() if ".attn." in n)


print(f"{'variant':<8}{'total':>10}{'attention':>12}")
for v in ("none", "SE", "ECA"):
    m = SmallCNN(v)
    print(f"{v:<8}{sum(p.numel() for p in m.parameters()):>10,}{attention_params(m):>12,}")

## 4. The comparison that is not a comparison

This is the part worth slowing down for.

`torch.manual_seed(seed)` before building each model looks like it makes the three variants comparable. It does not. Modules draw their parameters from one random stream in construction order, so inserting an SE block into the first block shifts the stream, and every convolution built after it gets different numbers.

Run the cell. Block 1's first convolution matches across variants, because it is built before any attention block. Block 2's does not. Half the network differs, and an ablation run that way measures the attention block plus an initialisation change, while reporting it as the attention block alone.

In [ ]:
def build_matched(variant, seed, widths=(32, 64), **kw):
    """A model whose backbone starts from weights identical across variants.

    Seeding alone does not do this. Modules draw their parameters from one
    stream in construction order, so inserting an SE block shifts the stream
    and every convolution built after it gets different numbers. The three
    variants would then differ by their attention block *and* by half their
    initial weights, which is not the comparison anyone means to run.
    """
    torch.manual_seed(seed)
    reference = SmallCNN("none", widths).state_dict()
    torch.manual_seed(seed)
    model = SmallCNN(variant, widths, **kw)
    state = model.state_dict()
    for key, value in state.items():
        if ".attn." not in key and key in reference and reference[key].shape == value.shape:
            state[key] = reference[key].clone()
    model.load_state_dict(state)
    return model


torch.manual_seed(0); naive_none = SmallCNN("none")
torch.manual_seed(0); naive_se = SmallCNN("SE")
print("same seed, no pairing:")
print(f"  block 1 conv 1 identical: {torch.equal(naive_none.b1.conv1.weight, naive_se.b1.conv1.weight)}")
print(f"  block 2 conv 1 identical: {torch.equal(naive_none.b2.conv1.weight, naive_se.b2.conv1.weight)}")
paired_none, paired_se = build_matched("none", 0), build_matched("SE", 0)
print("with build_matched:")
print(f"  block 2 conv 1 identical: {torch.equal(paired_none.b2.conv1.weight, paired_se.b2.conv1.weight)}")

## 5. The ablation

Three variants, three seeds, one shared initialisation, identical data order.

Read the per-seed differences before the means. The question is not whether the mean moved, it is whether the movement is larger than the spread between seeds of the same model. A gain that changes sign from seed to seed is not a gain.

In [ ]:
@torch.no_grad()
def accuracy(model, x, y, batch=1000):
    model.eval()
    return sum((model(x[i:i+batch]).argmax(1) == y[i:i+batch]).sum().item()
               for i in range(0, len(x), batch)) / len(x)


def train(variant, seed, epochs=EPOCHS, **kw):
    model = build_matched(variant, seed, **kw)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    steps = epochs * math.ceil(len(Xtr) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        model.train()
        for idx in torch.randperm(len(Xtr), generator=g).split(BATCH):
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()
            sched.step()
    return model, accuracy(model, Xte, yte)


SEEDS = (0, 1, 2)
t0 = time.perf_counter()
acc, models = {}, {}
for variant in ("none", "SE", "ECA"):
    acc[variant] = []
    for seed in SEEDS:
        model, a = train(variant, seed)
        acc[variant].append(a)
        if seed == 0:
            models[variant] = model
    print(f"  {variant:<5} {np.round(acc[variant], 4)}  mean {np.mean(acc[variant]):.4f} "
          f"[{time.perf_counter() - t0:.0f}s]", flush=True)

base = np.array(acc["none"])
print(f"\nbaseline spread across seeds: {base.max() - base.min():+.4f}")
for variant in ("SE", "ECA"):
    d = np.array(acc[variant]) - base
    print(f"{variant:<4} paired gain per seed {np.round(d, 4)}  mean {d.mean():+.4f}  "
          f"{'same sign on every seed' if (d > 0).all() or (d < 0).all() else 'SIGN FLIPS ACROSS SEEDS'}")

## 6. What the gate learned

The gate is a function of the input, so it can differ per image and per class. Averaging it by class shows whether the block learned to emphasise different channels for a sandal than for a pullover, or settled on one fixed rescaling that ignores the input.

Two numbers separate those cases: the spread across classes for a fixed channel, and the spread across channels averaged over classes. If the second is much larger than the first, the block learned a constant, and a constant per-channel scale is something the preceding batch norm could already do.

In [ ]:
@torch.no_grad()
def block2_gates(model, x):
    """The gate the second block produces, one value per channel."""
    h = model.b1(x)
    h = F.relu(model.b2.bn1(model.b2.conv1(h)))
    h = F.relu(model.b2.bn2(model.b2.conv2(h)))
    s = h.mean(dim=(2, 3))
    a = model.b2.attn
    if isinstance(a, SqueezeExcitation):
        return torch.sigmoid(a.fc2(F.relu(a.fc1(s))))
    if isinstance(a, EfficientChannelAttention):
        return torch.sigmoid(a.conv(s[:, None, :])).squeeze(1)
    return torch.ones_like(s)


fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, variant in zip(axes, ("SE", "ECA")):
    grid = np.stack([block2_gates(models[variant], Xte[yte == c][:300]).mean(0).numpy()
                     for c in range(10)])
    im = ax.imshow(grid, aspect="auto", cmap="YlGnBu", vmin=grid.min(), vmax=grid.max())
    ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES, fontsize=8)
    ax.set_xlabel("channel in block 2"); ax.set_title(f"{variant} gate, averaged by class")
    plt.colorbar(im, ax=ax)
    print(f"{variant}: gate range {grid.min():.3f} to {grid.max():.3f}, "
          f"spread across classes {grid.std(axis=0).mean():.4f}, "
          f"across channels {grid.mean(axis=0).std():.4f}")
plt.tight_layout(); plt.show()

## Exercises

1. **Break the pairing.** Swap `build_matched` for a plain `SmallCNN(variant)` after `torch.manual_seed(seed)` and re-run section 5. Compare the spread of the paired differences before and after. This is the cost of the mistake in section 4.
2. **Sweep the reduction ratio.** Run SE with `reduction` in 4, 8, 16, 32 and plot accuracy against attention parameters. The ratio is SE's only knob; find out whether it buys anything.
3. **Test the adaptive kernel.** ECA derives k from C rather than tuning it. Force `k` to 3, 5 and 9 and see whether the formula's choice wins.
4. **Where does the block go?** Move the attention to before the convolutions instead of after, and to after the pooling instead of before. Channel attention is usually described as position-independent, so it should not matter much. Check.
5. **Give it something to do.** Widen the second block to 128 or 256 channels and re-run. Both blocks reweight channels, so their scope grows with C, and the published results are on networks far wider than this one.


## Further reading

- Hu, Shen & Sun (2018), [Squeeze-and-Excitation Networks](https://arxiv.org/abs/1709.01507)
- Wang et al. (2020), [ECA-Net: Efficient Channel Attention for Deep Convolutional Neural Networks](https://arxiv.org/abs/1910.03151)
- Woo et al. (2018), [CBAM: Convolutional Block Attention Module](https://arxiv.org/abs/1807.06521), the spatial counterpart
- Bello et al. (2021), [Revisiting ResNets: Improved Training and Scaling Strategies](https://arxiv.org/abs/2103.07579), on how much of a reported architectural gain is the training recipe
- [Convolutional Neural Networks from Scratch](https://sesen.ai/blog/convolutional-neural-networks-from-scratch), the backbone this post attaches a block to
- [Vision Transformers from Scratch](https://sesen.ai/blog/vision-transformers-from-scratch), attention over positions rather than channels
